<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 Taller Hands-On: El Examen Sellado, la Moneda y la Llave Perdida 🔑
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Taller Práctico Evaluativo · Módulo 07
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/homeworks/Para%20Dummies/07_Seleccion_Modelos_Optimizacion_Hands_On_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a practicar? 🎈

Esta es la versión **guiada** 🧸 del [taller estándar del Módulo 07](../07_Seleccion_Modelos_Optimizacion_Hands_On.ipynb). El problema: una cooperativa quiere predecir si un **cliente caerá en mora** y tú tienes que **elegir y calificar** un modelo sin hacerte trampa. Repasamos las ideas de los cuadernos [Evaluación y selección](../../07%20-%20Seleccion%20de%20Modelos%20y%20Optimizacion/Para%20Dummies/00_Evaluacion_Seleccion_Dummies.ipynb) y [Random y Grid Search](../../07%20-%20Seleccion%20de%20Modelos%20y%20Optimizacion/Para%20Dummies/02_Random_Grid_Search_Dummies.ipynb):

1. 📝 **El examen final sellado** (Reto 1): un modelo memorión saca 10 en lo que ya vio y mucho menos en lo nuevo.
2. 🪙 **¿Suerte o mérito?** (Reto 2): la nota viene con un «margen de error» que puedes medir lanzando la moneda muchas veces (*bootstrap*).
3. 🔑 **La escala mágica** (Reto 3): buscar la mejor llave probando valores al azar, pero en escala de 10 en 10.

**Cómo trabajar:** completa los huecos `...` y los `# TODO` (siempre tienes una pista 💡), luego ejecuta la celda de **autoverificación**: si algo falta, te dirá qué revisar. Al final, *Restart & Run All*. No necesitas descargar nada.

---
### 📦 Los datos (ejecuta sin modificar)

600 solicitudes de crédito **ficticias** con 5 variables numéricas. `y = 1` significa **mora** (≈ 40 %).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import warnings; warnings.filterwarnings("ignore")

rng0 = np.random.default_rng(5)
X = pd.DataFrame(rng0.normal(size=(600, 5)), columns=["ingreso", "deuda", "antiguedad", "edad", "ahorro"])
beta = np.array([1.1, -0.9, 0.6, 0.0, 0.5])      # "edad" no aporta nada: es ruido
y = pd.Series((rng0.random(600) < 1 / (1 + np.exp(-(X.to_numpy() @ beta - 0.4)))).astype(int), name="mora")
print(X.shape, "| tasa de mora:", round(y.mean(), 3))
X.head()

---
## Reto 1: El Examen Final Sellado 📝🔐 *(25 puntos)*

**Analogía.** Un estudiante memorión se aprende las respuestas de la guía: en la guía saca 10, pero en un examen nuevo, ¡le va mucho peor! Para saberlo hay que guardar un **examen final sellado** (el conjunto de prueba) que el modelo nunca vea al entrenar.

**Qué hacer.**
1. Separa los datos: 25 % para la prueba, `stratify=y`, `random_state=0`.
2. Entrena un árbol **sin límite de profundidad** (el «memorión») y mide su *accuracy* en entrenamiento (`acc_train`) y en prueba (`acc_test`).
3. Calcula la **línea base** (`acc_base`): la *accuracy* en prueba de un `DummyClassifier(strategy="most_frequent")`, que siempre responde la clase más común.
4. Entrena una regresión logística (con `StandardScaler`) y guarda su *accuracy* en prueba en `acc_lr`.

**Meta.** El árbol debe sacar ≈ 100 % en entrenamiento y claramente menos en prueba; la regresión logística debe superar a la línea base.

In [ ]:
# 1) Separar: examen de práctica (dev) y examen final sellado (test)
X_dev, X_test, y_dev, y_test = train_test_split(..., ..., test_size=..., stratify=..., random_state=0)   # 💡 completa los huecos

# 2) El memorión: árbol sin límite de profundidad
arbol = DecisionTreeClassifier(random_state=0).fit(X_dev, y_dev)
acc_train = arbol.score(..., ...)    # 💡 ¿con qué datos lo evalúas para ver cuánto memorizó?
acc_test = arbol.score(..., ...)     # 💡 ¿y con cuáles para ver cuánto aprendió de verdad?

# 3) Línea base: siempre responde la clase más común
base = DummyClassifier(strategy="most_frequent").fit(X_dev, y_dev)
acc_base = None  # TODO: base.score(...) sobre la prueba

# 4) Modelo sencillo: escalador + regresión logística
lr = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_dev, y_dev)
acc_lr = None  # TODO: lr.score(...) sobre la prueba

print(f"Árbol: entrenamiento {acc_train} | prueba {acc_test}")
print(f"Línea base: {acc_base} | Regresión logística: {acc_lr}")

In [ ]:
# ✅ Autoverificación del Reto 1 (no modifiques esta celda)
_falta = [v for v in ("X_dev", "X_test", "y_dev", "y_test", "acc_train", "acc_test", "acc_base", "acc_lr",) if v not in globals() or globals()[v] is None or globals()[v] is Ellipsis]
assert not _falta, f"💡 Reto 1: todavía faltan por crear/completar: {_falta}. Reemplaza los huecos '...' y los TODO."
assert X_dev.shape == (450, 5) and X_test.shape == (150, 5), "💡 Reto 1: la partición debe dejar 450 filas para entrenar y 150 para la prueba (test_size=0.25)."
assert acc_train > 0.99, "💡 Reto 1: el árbol sin límite de profundidad memoriza: su accuracy en ENTRENAMIENTO debería ser ≈ 1. ¿Usaste X_dev, y_dev?"
assert acc_test < acc_train - 0.10, "💡 Reto 1: en la PRUEBA el memorión debería sacar al menos 10 puntos menos. ¿Usaste X_test, y_test?"
assert acc_base is not None and 0.5 <= acc_base <= 0.7, "💡 Reto 1: la línea base 'most_frequent' debería acertar la proporción de la clase mayoritaria (≈ 0.6). Calcula acc_base con la prueba."
assert acc_lr is not None and acc_lr > acc_base + 0.08, "💡 Reto 1: la regresión logística debería superar a la línea base por al menos 8 puntos; evalúa lr sobre X_test, y_test."
print("✅ Reto 1 superado: ya sabes que la nota de entrenamiento engaña.")

---
## Reto 2: ¿Suerte o Mérito? El Margen de Error con la Moneda 🪙 *(25 puntos)*

**Analogía.** Si lanzas una moneda 10 veces y salen 6 caras, no concluyes que está cargada: ¡es suerte! Lo mismo pasa con la *accuracy*: es **una muestra** y cambiaría si el examen tuviera otras preguntas. El *bootstrap* lo simula: «rehacemos el examen» muchas veces **sacando preguntas al azar con reemplazo** y vemos cuánto varía la nota.

**Qué hacer** (con el modelo `lr` del Reto 1):
1. Obtén las predicciones `pred = lr.predict(X_test)` y el vector `ok` de aciertos (1 si acertó, 0 si falló).
2. Repite **200 veces**: elige 150 posiciones al azar **con reemplazo** (usa `rng.integers(0, 150, 150)`) y calcula la *accuracy* de ese remuestreo (el promedio de `ok`). Guarda las 200 notas en `notas`.
3. Calcula el intervalo del 95 %: percentiles 2.5 y 97.5 de `notas` (`ic_inf`, `ic_sup`).

**Meta.** Tu nota `acc_lr` debe quedar dentro del intervalo, que no debe ser ni un punto ni enorme (entre 0 y 0.20 de ancho).

In [ ]:
rng = np.random.default_rng(0)

pred = lr.predict(X_test)
ok = (pred == y_test.to_numpy()).astype(int)     # 1 si acertó, 0 si falló

notas = []
for _ in range(200):
    idx = rng.integers(0, 150, 150)              # 💡 150 preguntas al azar, CON reemplazo
    notas.append(...)                            # 💡 la accuracy del remuestreo es el promedio de ok[idx]

ic_inf, ic_sup = np.percentile(notas, [..., ...])  # 💡 los percentiles del 95 % central
print(f"Nota del modelo: {acc_lr:.3f} | Intervalo 95 %: ({ic_inf}, {ic_sup})")

In [ ]:
# ✅ Autoverificación del Reto 2 (no modifiques esta celda)
_falta = [v for v in ("notas", "ic_inf", "ic_sup",) if v not in globals() or globals()[v] is None or globals()[v] is Ellipsis]
assert not _falta, f"💡 Reto 2: todavía faltan por crear/completar: {_falta}. Reemplaza los huecos '...' y los TODO."
assert len(notas) == 200, "💡 Reto 2: debes guardar 200 notas (una por remuestreo) en la lista 'notas'."
assert all(isinstance(v, (int, float, np.floating)) for v in notas), "💡 Reto 2: cada nota debe ser un número (el promedio de ok[idx]), no un hueco '...'."
assert 0.5 < np.mean(notas) < 1.0, "💡 Reto 2: el promedio de 'notas' debería parecerse a la accuracy del modelo (≈ 0.75-0.85)."
assert isinstance(ic_inf, (float, np.floating)) and ic_inf < ic_sup, "💡 Reto 2: ic_inf debe ser menor que ic_sup; usa percentiles 2.5 y 97.5."
assert ic_inf <= acc_lr <= ic_sup, "💡 Reto 2: la nota del modelo debe caer dentro del intervalo; revisa que remuestreas 150 posiciones con reemplazo."
assert 0.0 < ic_sup - ic_inf < 0.20, f"💡 Reto 2: el ancho ({ic_sup - ic_inf:.3f}) debería estar entre 0 y 0.20."
print(f"✅ Reto 2 superado: tu nota de {acc_lr:.2f} viene con un margen de ± {(ic_sup - ic_inf) / 2:.2f}.")

---
## Reto 3: La Escala Mágica y la Búsqueda al Azar 🔑 *(25 puntos)*

**Analogía.** Un radio con una perilla que va de 0.001 a 100: si pruebas de uno en uno, ¡casi todo tu esfuerzo cae entre 50 y 100! La **escala mágica** prueba **de 10 en 10** (0.001, 0.01, 0.1, 1, 10, 100). Para eso sorteamos el **exponente**, no el número.

**Qué hacer** (SVM con `C` como perilla):
1. Sortea 12 exponentes con `rng.uniform(-3, 2, 12)` y conviértelos: `valores_C = 10 ** exponentes`.
2. Para cada `C` calcula la *accuracy* media con `cross_val_score(make_pipeline(StandardScaler(), SVC(C=C)), X_dev, y_dev, cv=3)` y guárdala en `puntajes`.
3. Escoge la mejor perilla: `mejor_C` (el `C` con mayor puntaje) y `mejor_puntaje`.

**Meta.** 12 valores entre 0.001 y 100 que abarquen varios órdenes de magnitud, y un `mejor_puntaje` mayor que la línea base.

In [ ]:
rng = np.random.default_rng(1)

exponentes = rng.uniform(-3, 2, 12)
valores_C = ...                      # 💡 10 elevado a cada exponente

puntajes = []
for C in valores_C:
    modelo = make_pipeline(StandardScaler(), SVC(C=C))
    puntajes.append(cross_val_score(modelo, ..., ..., cv=3).mean())   # 💡 datos de desarrollo

mejor_C = None        # TODO: el valor de valores_C con el puntaje más alto (pista: np.argmax)
mejor_puntaje = None  # TODO: ese puntaje más alto
print("Mejor C:", mejor_C, "| mejor puntaje:", mejor_puntaje)

In [ ]:
# ✅ Autoverificación del Reto 3 (no modifiques esta celda)
_falta = [v for v in ("valores_C", "puntajes", "mejor_C", "mejor_puntaje",) if v not in globals() or globals()[v] is None or globals()[v] is Ellipsis]
assert not _falta, f"💡 Reto 3: todavía faltan por crear/completar: {_falta}. Reemplaza los huecos '...' y los TODO."
assert len(valores_C) == 12, "💡 Reto 3: valores_C debe tener 12 valores: 10 ** exponentes."
assert min(valores_C) >= 1e-3 and max(valores_C) <= 1e2, "💡 Reto 3: los exponentes están en [-3, 2], así que C debe quedar entre 0.001 y 100."
assert max(valores_C) / min(valores_C) > 50, "💡 Reto 3: con la escala mágica los valores deben abarcar varios órdenes de magnitud (¿elevaste 10 al exponente?)."
assert len(puntajes) == 12 and all(0.4 < p <= 1 for p in puntajes), "💡 Reto 3: puntajes debe tener 12 accuracies (entre 0 y 1) calculadas con cross_val_score sobre X_dev, y_dev."
assert mejor_puntaje is not None and abs(mejor_puntaje - max(puntajes)) < 1e-12, "💡 Reto 3: mejor_puntaje es el máximo de 'puntajes'."
assert mejor_C is not None and abs(mejor_C - valores_C[int(np.argmax(puntajes))]) < 1e-12, "💡 Reto 3: mejor_C es el C que obtuvo el mejor puntaje (pista: np.argmax)."
assert mejor_puntaje > acc_base + 0.05, "💡 Reto 3: el mejor SVM debería superar a la línea base (acc_base) por al menos 5 puntos."
print(f"✅ Reto 3 superado: la mejor perilla fue C = {mejor_C:.3g}.")

---
## Reto 4: Mini-Reflexión ✍️ *(25 puntos)*

Escribe **3 o 4 líneas** con tus palabras (no hace falta código): ¿Por qué el árbol memorión sacó mucho mejor nota en entrenamiento que en la prueba? ¿Qué te dice el intervalo del Reto 2 sobre confiar en una sola nota? ¿Por qué buscar `C` «de 10 en 10» es mejor que «de 1 en 1»? 🧸

#### ✍️ Tu respuesta

_Escribe aquí tus 3 o 4 líneas. Reemplaza este texto por tu respuesta._

---
### 🧠 Rúbrica de evaluación

| Criterio | Puntos |
|---|---|
| Reto 1: partición, memorión frente a línea base | 25 |
| Reto 2: intervalo *bootstrap* de la accuracy | 25 |
| Reto 3: búsqueda al azar en escala logarítmica | 25 |
| Reto 4: mini-reflexión con tus palabras | 25 |

### ✅ Checklist de entrega

- [ ] No quedan `...` ni `# TODO` sin completar.
- [ ] Las tres autoverificaciones imprimen «✅ Reto N superado».
- [ ] Hiciste *Restart & Run All* y todo corrió sin errores.

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Machine Learning (Edición Didáctica Para Dummies)</i>
  </p>
</div>